In [6]:
# In een notebook cel:
%pip install pettingzoo[atari] multi-agent-ale-py autorom[accept-rom-license]
%pip install tensorflow tensorflow-probability numpy matplotlib scikit-learn pandas

# Download Atari ROMs (alleen de eerste keer nodig)
%AutoROM -y

  Using cached multi_agent_ale_py-0.1.12.tar.gz (552 kB)
  Installing build dependencies: started
  Installing build dependencies: still running...
  Installing build dependencies: still running...
  Installing build dependencies: still running...
  Installing build dependencies: still running...
  Installing build dependencies: still running...
  Installing build dependencies: still running...
  Installing build dependencies: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × installing build dependencies for multi-agent-ale-py did not run successfully.
  │ exit code: 1
  ╰─> [406 lines of output]
        Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
        Using cached wheel-0.47.0-py3-none-any.whl.metadata (2.3 kB)
        Using cached cmake-4.3.4-py3-none-win_amd64.whl.metadata (6.5 kB)
        Using cached ninja-1.13.0-py3-none-win_amd64.whl.metadata (5.1 kB)
        Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
      Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
      Using cached wheel-0.47.0-py3-none-any.whl (32 kB)
         ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
         ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
         ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
         ---------------------------------------- 0.3/41.3 MB ? eta -:--:--
          ----------------------------------

Note: you may need to restart the kernel to use updated packages.


UsageError: Line magic function `%AutoROM` not found.


In [7]:
# Importeer libraries
from pettingzoo.atari import warlords_v3
from A2C_agent import A2CAgent
import numpy as np
import matplotlib.pyplot as plt

# Maak environment aan (RAM mode)
env = warlords_v3.env(obs_type="ram")

# Reset environment om shapes te bepalen
obs, _ = env.reset()
state_dim = obs.shape  # (128,) voor RAM
action_dim = env.action_space(env.agents[0]).n  # Aantal acties (meestal 18)

print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Agents: {env.agents}")

ModuleNotFoundError: No module named 'multi_agent_ale_py'

In [ ]:
# Maak A2C agent aan met hyperparameters
a2c_agent = A2CAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    player_id=0,  # Of de juiste player_id voor jouw agent
    learning_rate=0.001,
    gamma=0.99,
    entropy_coef=0.01,
    value_coef=0.5,
    max_grad_norm=0.5,
    hidden_layers=[256, 128]
)

print("A2C Agent geïnitialiseerd!")

In [ ]:
# Training parameters
num_episodes = 500
eval_interval = 50  # Evalueer elke X episodes
save_interval = 100  # Sla model op elke X episodes

# Training loop
print(f"Start training voor {num_episodes} episodes...")

for episode in range(num_episodes):
    # Train één episode
    episode_reward = a2c_agent.train_episode(env, "A2C")
    
    # Print voortgang
    if (episode + 1) % 10 == 0:
        avg_reward = np.mean(a2c_agent.episode_rewards[-10:])
        print(f"Episode {episode+1}/{num_episodes} | "
              f"Reward: {episode_reward:.2f} | "
              f"Avg (last 10): {avg_reward:.2f} | "
              f"Total Episodes: {len(a2c_agent.episode_rewards)}")
    
    # Evalueer periodiek
    if (episode + 1) % eval_interval == 0:
        mean_reward, std_reward = a2c_agent.evaluate(env, n_episodes=10)
        print(f"  --- Evaluation: {mean_reward:.2f} ± {std_reward:.2f} ---")
    
    # Sla model op periodiek
    if (episode + 1) % save_interval == 0:
        a2c_agent.actor.save_weights(f'a2c_actor_ep{episode+1}.h5')
        a2c_agent.critic.save_weights(f'a2c_critic_ep{episode+1}.h5')
        print(f"  --- Model saved at episode {episode+1} ---")

print("Training voltooid!")

In [ ]:
# Finale evaluatie
final_mean, final_std = a2c_agent.evaluate(env, n_episodes=20)
print(f"\nFinale Evaluatie:")
print(f"Mean Reward: {final_mean:.2f} ± {final_std:.2f}")
print(f"Total training episodes: {len(a2c_agent.episode_rewards)}")

In [ ]:
def plot_training_curves(agent):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Rewards
    rewards = agent.episode_rewards
    axes[0, 0].plot(rewards, alpha=0.3, label='Raw Rewards')
    
    # Moving average
    window = 20
    if len(rewards) >= window:
        moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
        axes[0, 0].plot(range(window-1, len(rewards)), moving_avg, 
                        label=f'Moving Avg (window={window})', linewidth=2)
    
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Reward')
    axes[0, 0].set_title('Training Rewards')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Losses
    if agent.episode_losses:
        actor_losses = [loss[0] for loss in agent.episode_losses]
        critic_losses = [loss[1] for loss in agent.episode_losses]
        
        axes[0, 1].plot(actor_losses, label='Actor Loss', alpha=0.7)
        axes[0, 1].plot(critic_losses, label='Critic Loss', alpha=0.7)
        axes[0, 1].set_xlabel('Episode')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].set_title('Training Losses')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
    
    # Reward distribution
    axes[1, 0].hist(rewards, bins=30, alpha=0.7, edgecolor='black')
    axes[1, 0].set_xlabel('Reward')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Reward Distribution')
    axes[1, 0].grid(True)
    
    # Training stability (rolling std)
    if len(rewards) >= window:
        rolling_std = []
        for i in range(window, len(rewards)+1):
            rolling_std.append(np.std(rewards[i-window:i]))
        axes[1, 1].plot(range(window, len(rewards)+1), rolling_std)
        axes[1, 1].set_xlabel('Episode')
        axes[1, 1].set_ylabel('Rolling Std Dev')
        axes[1, 1].set_title(f'Training Stability (window={window})')
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()

# Plot de resultaten
plot_training_curves(a2c_agent)

In [ ]:
# Sla het finale model op
a2c_agent.actor.save_weights('a2c_actor_final.h5')
a2c_agent.critic.save_weights('a2c_critic_final.h5')

# Sla ook de hyperparameters op
import json
hyperparams = {
    'learning_rate': 0.001,
    'gamma': 0.99,
    'entropy_coef': 0.01,
    'value_coef': 0.5,
    'max_grad_norm': 0.5,
    'hidden_layers': [256, 128],
    'num_episodes': len(a2c_agent.episode_rewards),
    'final_mean_reward': final_mean,
    'final_std_reward': final_std
}

with open('a2c_hyperparams.json', 'w') as f:
    json.dump(hyperparams, f, indent=4)

print("Model en hyperparameters opgeslagen!")

In [ ]:
# Laad een getraind model
loaded_agent = A2CAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    player_id=0,
    learning_rate=0.001,
    gamma=0.99,
    entropy_coef=0.01,
    hidden_layers=[256, 128]
)

# Laad de weights
loaded_agent.actor.load_weights('a2c_actor_final.h5')
loaded_agent.critic.load_weights('a2c_critic_final.h5')

print("Model geladen!")

In [ ]:
# In het tournament notebook, vervang de PPO agent import:
from A2C_agent import A2CAgent

# Laad het getrainde model
a2c_trained = A2CAgent(
    state_dim=(128,),
    action_dim=18,
    player_id=0,
    learning_rate=0.001,
    gamma=0.99,
    entropy_coef=0.01,
    hidden_layers=[256, 128]
)

a2c_trained.actor.load_weights('a2c_actor_final.h5')
a2c_trained.critic.load_weights('a2c_critic_final.h5')

# Voeg toe aan agent_instances
agent_instances = [
    RandomAgent(),
    HeuristicAgent(),
    a2c_trained,  # Je getrainde A2C agent
    RandomAgent()
]

agent_names = ['RandomAgent', 'HeuristicAgent', 'A2CAgent', 'Agent4']